# RAG Evaluation with EvalHub and Ragas

This notebook demonstrates how to evaluate a RAG pipeline using **Ragas** (Retrieval Augmented Generation Assessment) through the **EvalHub SDK**.

Ragas evaluates RAG pipeline quality - not raw LLM performance. It uses the LLM as a judge to score how well the system retrieves context and generates grounded answers.

| Aspect | lm_evaluation_harness | ragas |
|---|---|---|
| Evaluates | Raw LLM text generation | RAG pipeline outputs |
| Input data | Built-in benchmarks (ARC, MMLU) | User-provided JSONL with RAG Q&A |
| Judge model | N/A (compares vs ground truth) | Uses the LLM itself as judge |
| Key question | "How smart is this LLM?" | "Is my RAG grounding answers correctly?" |

**Why not the UI?** The RHOAI 3.5 dashboard doesn't expose the `test_data_ref` field (S3 dataset source) for Ragas. The UI was designed for `lm_evaluation_harness` which has built-in benchmarks. We submit via the SDK instead - results still appear in the dashboard.

## Prerequisites

1. EvalHub deployed with `ragas` in the providers list
2. Ragas dataset uploaded to MinIO/S3 (JSONL with `user_input`, `response`, `retrieved_contexts`, `reference`)
3. S3 credentials secret in the tenant namespace
4. Model auth secret for the judge LLM
5. Environment variables set (injected by K8s secrets on OpenShift, or via `.env` / shell export locally):
   - `EVALHUB_BASE_URL` - EvalHub API endpoint
   - `EVALHUB_TOKEN` - OpenShift bearer token
   - `EVALHUB_TENANT` - Tenant namespace (e.g. `wskp-user1`)

## Install Dependencies

In [ ]:
%pip install -q "eval-hub-sdk[client]" pyyaml boto3 pandas tabulate

## Configuration

In [ ]:
import os
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# -- EvalHub connection --
os.environ.setdefault("EVALHUB_BASE_URL", "https://evalhub.apps.cluster.example.com")
os.environ.setdefault("EVALHUB_TOKEN", "")  # oc whoami -t
os.environ.setdefault("EVALHUB_TENANT", "wskp-user1")

# -- Judge model --
MODEL_URL = os.environ.get("MODEL_URL", "https://litemaas.rhoai.rh-aiservices-bu.com")
MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen3.6-35B-A3B")
MODEL_AUTH_SECRET = os.environ.get("MODEL_AUTH_SECRET", "qwen36-35b-a3b-api-key")

# -- S3 / MinIO --
S3_CREDENTIALS_SECRET = os.environ.get("S3_CREDENTIALS_SECRET", "eval-s3-credentials")
S3_BUCKET = os.environ.get("S3_BUCKET", "eval-data")
S3_KEY = os.environ.get("S3_KEY", "ragas")

# -- Ragas config --
BENCHMARK_ID = "ragas_rag_default"
METRICS = ["faithfulness", "context_precision", "context_recall"]
MAX_TOKENS = 4096
TEMPERATURE = 0.1
MAX_WORKERS = 2

# Verify required vars
required = ["EVALHUB_BASE_URL", "EVALHUB_TOKEN"]
missing = [v for v in required if not os.environ.get(v)]
if missing:
    print(f"WARNING: Missing environment variables: {', '.join(missing)}")
    print("Set EVALHUB_TOKEN with: oc whoami -t")
else:
    print("Configuration OK")

print(f"EvalHub URL:  {os.environ['EVALHUB_BASE_URL']}")
print(f"Tenant:       {os.environ['EVALHUB_TENANT']}")
print(f"Judge model:  {MODEL_NAME} @ {MODEL_URL}")
print(f"S3 dataset:   s3://{S3_BUCKET}/{S3_KEY}/")
print(f"Metrics:      {METRICS}")

## Connect to EvalHub

In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url=os.environ["EVALHUB_BASE_URL"],
    auth_token=os.environ["EVALHUB_TOKEN"],
)

health = client.health()
print(f"EvalHub status: {health}")

## 1. Inspect the Dataset

The Ragas dataset is a JSONL file with four columns per record:
- `user_input` - question asked to the RAG system
- `response` - what the RAG system answered
- `retrieved_contexts` - array of context passages the retriever found
- `reference` - ground truth reference answer

This dataset was adapted from Fed Aura Capital mortgage lending Q&A pairs. Question 8 intentionally has an incorrect response (says DTI 45% instead of 40%) to test faithfulness detection.

In [ ]:
import pandas as pd

dataset_path = Path("data/ragas-fedaura-testdata.jsonl")

if dataset_path.exists():
    records = [json.loads(line) for line in dataset_path.read_text().strip().splitlines()]
    df = pd.DataFrame(records)
    print(f"Dataset: {len(records)} records")
    print(f"Columns: {list(df.columns)}")
    print()

    # Preview first few questions
    for i, row in enumerate(records[:5], 1):
        q = row["user_input"][:80]
        a = row["response"][:60]
        n_ctx = len(row["retrieved_contexts"])
        print(f"{i}. {q}")
        print(f"   Response: {a}...")
        print(f"   Contexts: {n_ctx} passage(s)")
        print()
else:
    print(f"Dataset not found at {dataset_path}")
    print("Make sure the dataset is uploaded to S3 - this preview is optional.")

## 2. Upload Dataset to S3 (Optional)

Skip this cell if the dataset is already in MinIO. This requires port-forwarding MinIO locally.

In [ ]:
# Uncomment to upload dataset to MinIO
# Requires: oc port-forward svc/minio-service -n wskp-user1 9099:9000 &

# import boto3
# s3 = boto3.client(
#     "s3",
#     endpoint_url="http://localhost:9099",
#     aws_access_key_id="minio",
#     aws_secret_access_key="<your-secret>",
#     region_name="us-east-1",
# )
#
# # Create bucket if needed
# try:
#     s3.create_bucket(Bucket=S3_BUCKET)
#     print(f"Created bucket: {S3_BUCKET}")
# except s3.exceptions.BucketAlreadyOwnedByYou:
#     print(f"Bucket exists: {S3_BUCKET}")
#
# # Upload
# s3.upload_file(str(dataset_path), S3_BUCKET, f"{S3_KEY}/dataset.jsonl")
# print(f"Uploaded to s3://{S3_BUCKET}/{S3_KEY}/dataset.jsonl")

## 3. Submit Ragas Evaluation

Submit the evaluation job via the EvalHub Python SDK. The job runs on the cluster as a pod with the Ragas adapter container, using the judge LLM to score each record.

In [ ]:
from evalhub import ModelConfig
from evalhub.models.api import BenchmarkConfig, JobSubmissionRequest

benchmark = BenchmarkConfig(
    id=BENCHMARK_ID,
    provider_id="ragas",
    parameters={
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "max_workers": MAX_WORKERS,
        "metrics": METRICS,
    },
    test_data_ref={
        "s3": {
            "bucket": S3_BUCKET,
            "key": S3_KEY,
            "secret_ref": S3_CREDENTIALS_SECRET,
        }
    },
)

request = JobSubmissionRequest(
    name="rag-quality-eval",
    model=ModelConfig(
        url=MODEL_URL,
        name=MODEL_NAME,
        auth={"secret_ref": MODEL_AUTH_SECRET},
    ),
    benchmarks=[benchmark],
)

print("Submitting Ragas evaluation job...")
print(f"  Model:     {MODEL_NAME}")
print(f"  Benchmark: {BENCHMARK_ID}")
print(f"  Metrics:   {METRICS}")
print(f"  Dataset:   s3://{S3_BUCKET}/{S3_KEY}/")
print()

tenant = os.environ["EVALHUB_TENANT"]
job = client.jobs.submit(request, tenant=tenant)

print(f"Job submitted!")
print(f"  Job ID: {job.id}")
print(f"  Status: {job.state}")

## 4. Monitor Job Progress

Ragas evaluation typically takes 5-15 minutes depending on dataset size and model throughput. The cell below polls until completion.

In [ ]:
import time

print(f"Waiting for job {job.id} to complete...")
print()

try:
    final = client.jobs.wait_for_completion(job.id, timeout=900, poll_interval=10)
    print(f"Job finished: {final.state}")
except TimeoutError:
    final = client.jobs.get(job.id, tenant=tenant)
    print(f"Timeout reached. Current status: {final.state}")
    print("Re-run this cell to keep polling, or check the RHOAI dashboard.")

## 5. View Results

Fetch and display the evaluation results. Scores are 0-1 where:
- **faithfulness** - is the response faithful to the retrieved context? (0.88+ means the RAG is grounding well)
- **context_precision** - did the retriever rank relevant documents higher?
- **context_recall** - did the retriever find all relevant information?

In [ ]:
# Get the job with results
result_job = client.jobs.get(job.id, tenant=tenant)

print(f"Job: {result_job.id}")
print(f"State: {result_job.state}")
print()

if hasattr(result_job, "benchmarks") and result_job.benchmarks:
    for bm in result_job.benchmarks:
        print(f"Benchmark: {bm.id}")
        if hasattr(bm, "results") and bm.results:
            results_data = []
            for r in bm.results:
                name = getattr(r, "name", getattr(r, "metric", "unknown"))
                score = getattr(r, "score", getattr(r, "value", "N/A"))
                results_data.append({"Metric": name, "Score": score})
            
            results_df = pd.DataFrame(results_data)
            print(results_df.to_string(index=False))
        else:
            print("No results yet - job may still be processing.")
else:
    print("No benchmark results available.")
    print("Check the RHOAI dashboard or pod logs for details.")

## 6. Check Adapter Logs (Optional)

If something goes wrong, check the Ragas adapter container logs directly.

In [ ]:
# Stream logs from the adapter container
# Uncomment to watch live logs:

# from evalhub import JobLogOptions
#
# for update in client.jobs.watch_logs(
#     job.id,
#     options=JobLogOptions(tail_lines=100),
#     poll_interval=5.0,
# ):
#     if update.logs:
#         print(update.logs, end="")

# Or via oc CLI (run in terminal):
# oc logs <pod-name> -n wskp-user1 -c adapter --tail=50

## Alternative: Submit via curl

If you prefer curl or can't install the SDK, use the REST API directly.

```bash
TOKEN=$(oc whoami -t)

curl -sk -X POST \
  "https://<evalhub-route>/api/v1/evaluations/jobs" \
  -H "Authorization: Bearer $TOKEN" \
  -H "Content-Type: application/json" \
  -H "X-Tenant: wskp-user1" \
  -H "X-User: admin" \
  -d '{
    "name": "rag-quality-eval",
    "model": {
      "url": "https://litemaas.rhoai.rh-aiservices-bu.com",
      "name": "Qwen3.6-35B-A3B",
      "auth": {"secret_ref": "qwen36-35b-a3b-api-key"}
    },
    "benchmarks": [{
      "id": "ragas_rag_default",
      "provider_id": "ragas",
      "parameters": {
        "temperature": 0.1,
        "max_tokens": 4096,
        "max_workers": 2,
        "metrics": ["faithfulness", "context_precision", "context_recall"]
      },
      "test_data_ref": {
        "s3": {
          "bucket": "eval-data",
          "key": "ragas",
          "secret_ref": "eval-s3-credentials"
        }
      }
    }]
  }'
```

Or use `envsubst` with the eval template:

```bash
export MODEL_URL="https://litemaas.rhoai.rh-aiservices-bu.com"
export MODEL_NAME="Qwen3.6-35B-A3B"
export MODEL_AUTH_SECRET="qwen36-35b-a3b-api-key"
export S3_CREDENTIALS_SECRET="eval-s3-credentials"

envsubst < eval-ragas-rag.yaml | evalhub eval run --config -
```

## Known Issues

1. **Metrics needing embeddings**: `answer_relevancy` and `semantic_similarity` require `/v1/embeddings`. If your endpoint doesn't support it, exclude them from the metrics list.

2. **IncompleteOutputException**: Ragas uses `instructor` for structured output extraction. Set `max_tokens: 4096` or higher to avoid truncated responses.

3. **Container image v0.5.0 bug**: The `v0.5.0` ragas adapter image has an `EvalHubOpenAILLM` vs `InstructorLLM` incompatibility. Use the `latest` image. Patch with:
   ```bash
   oc get cm evalhub-provider-ragas -n redhat-ods-applications -o json | \
     python3 -c "import sys,json; d=json.load(sys.stdin); d['data']['ragas.yaml']=d['data']['ragas.yaml'].replace('v0.5.0','latest'); json.dump(d,sys.stdout)" | \
     oc replace -f -
   ```

4. **TLS with ExternalName services**: Use the actual external hostname (e.g. `https://litemaas.rhoai.rh-aiservices-bu.com`), not the k8s service FQDN.

5. **TrustyAI operator reconciliation**: The operator may revert ConfigMap changes. Scale it down (`replicas=0`) before patching, then scale back up.

In [ ]:
# Cleanup
client.close()
print("Done.")